In [7]:
pi = {}
pi[1] = 2
pi[3] = 4
pi[5] = 6
pi[7] = 8
print(pi)
top_nodes = sorted(pi.items(), key=lambda x: x[1], reverse=True)[:5]
top_nodes = dict(top_nodes)
top_nodes, list(top_nodes.keys())

{1: 2, 3: 4, 5: 6, 7: 8}


({7: 8, 5: 6, 3: 4, 1: 2}, [7, 5, 3, 1])

In [1]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: d:\Documents\GitHub\NodeRAG\graphs
The root directory is: d:\Documents\GitHub\NodeRAG


In [2]:
import sys
sys.path.append(root_path)
from graphs.Node import Node
from graphs.prompt.entity_matching_prompt import entity_matching_prompt

In [3]:
#LLM api call
from google import genai

with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()

def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text,
        config={
            'thinking_config': {'thinking_budget': 0}
        }
    )
    return response.text, response.usage_metadata.total_token_count

In [15]:
import json
l = [
    'STAGE 4 CANCER', 'STAGE OF THE CANCER',
   'STAGE IV ANAL CANCER',
   'STAGE 4 BREAST CANCER',
   'CANCER STAGE',
   'STAGE IV (4)',
   'STAGE 4',
   'STAGE IA1 CANCER',
]
tokens = []
synonyms = {}
for e in l:
    if e in synonyms:
        continue
    prompt = entity_matching_prompt(e, l)
    value, token = call_gemini(prompt)
    value = json.loads(value.strip())
    for v in value:
        synonyms[v] = value
    tokens.append(token)
    print(synonyms)
    print("-"*50)

{'STAGE 4 CANCER': ['STAGE 4 CANCER', 'STAGE OF THE CANCER', 'STAGE IV ANAL CANCER', 'STAGE 4 BREAST CANCER', 'CANCER STAGE', 'STAGE IV (4)', 'STAGE 4', 'STAGE IA1 CANCER'], 'STAGE OF THE CANCER': ['STAGE 4 CANCER', 'STAGE OF THE CANCER', 'STAGE IV ANAL CANCER', 'STAGE 4 BREAST CANCER', 'CANCER STAGE', 'STAGE IV (4)', 'STAGE 4', 'STAGE IA1 CANCER'], 'STAGE IV ANAL CANCER': ['STAGE 4 CANCER', 'STAGE OF THE CANCER', 'STAGE IV ANAL CANCER', 'STAGE 4 BREAST CANCER', 'CANCER STAGE', 'STAGE IV (4)', 'STAGE 4', 'STAGE IA1 CANCER'], 'STAGE 4 BREAST CANCER': ['STAGE 4 CANCER', 'STAGE OF THE CANCER', 'STAGE IV ANAL CANCER', 'STAGE 4 BREAST CANCER', 'CANCER STAGE', 'STAGE IV (4)', 'STAGE 4', 'STAGE IA1 CANCER'], 'CANCER STAGE': ['STAGE 4 CANCER', 'STAGE OF THE CANCER', 'STAGE IV ANAL CANCER', 'STAGE 4 BREAST CANCER', 'CANCER STAGE', 'STAGE IV (4)', 'STAGE 4', 'STAGE IA1 CANCER'], 'STAGE IV (4)': ['STAGE 4 CANCER', 'STAGE OF THE CANCER', 'STAGE IV ANAL CANCER', 'STAGE 4 BREAST CANCER', 'CANCER STA

In [16]:
print("Token Count")
print(f"Min: {min(tokens)}, Max: {max(tokens)}, Sum: {sum(tokens)}, Avg: {sum(tokens)/len(tokens)}")
print("Num calls:", len(tokens))

Token Count
Min: 583, Max: 583, Sum: 583, Avg: 583.0
Num calls: 1


In [17]:
synonyms

{'STAGE 4 CANCER': ['STAGE 4 CANCER',
  'STAGE OF THE CANCER',
  'STAGE IV ANAL CANCER',
  'STAGE 4 BREAST CANCER',
  'CANCER STAGE',
  'STAGE IV (4)',
  'STAGE 4',
  'STAGE IA1 CANCER'],
 'STAGE OF THE CANCER': ['STAGE 4 CANCER',
  'STAGE OF THE CANCER',
  'STAGE IV ANAL CANCER',
  'STAGE 4 BREAST CANCER',
  'CANCER STAGE',
  'STAGE IV (4)',
  'STAGE 4',
  'STAGE IA1 CANCER'],
 'STAGE IV ANAL CANCER': ['STAGE 4 CANCER',
  'STAGE OF THE CANCER',
  'STAGE IV ANAL CANCER',
  'STAGE 4 BREAST CANCER',
  'CANCER STAGE',
  'STAGE IV (4)',
  'STAGE 4',
  'STAGE IA1 CANCER'],
 'STAGE 4 BREAST CANCER': ['STAGE 4 CANCER',
  'STAGE OF THE CANCER',
  'STAGE IV ANAL CANCER',
  'STAGE 4 BREAST CANCER',
  'CANCER STAGE',
  'STAGE IV (4)',
  'STAGE 4',
  'STAGE IA1 CANCER'],
 'CANCER STAGE': ['STAGE 4 CANCER',
  'STAGE OF THE CANCER',
  'STAGE IV ANAL CANCER',
  'STAGE 4 BREAST CANCER',
  'CANCER STAGE',
  'STAGE IV (4)',
  'STAGE 4',
  'STAGE IA1 CANCER'],
 'STAGE IV (4)': ['STAGE 4 CANCER',
  'STAGE

In [18]:
unique = [list(v) for v in {tuple(v) for v in synonyms.values()}]
unique

[['STAGE 4 CANCER',
  'STAGE OF THE CANCER',
  'STAGE IV ANAL CANCER',
  'STAGE 4 BREAST CANCER',
  'CANCER STAGE',
  'STAGE IV (4)',
  'STAGE 4',
  'STAGE IA1 CANCER']]

In [8]:
import pickle
with open(f"{root_path}/graphs/data/nodes/entity/medical_entities.pkl", "rb") as f:
    medical_entities = pickle.load(f)

In [9]:
len(medical_entities)

9100